# Task 2 — Prompt Engineering & AI Productivity Toolkit
**Module 02 | Day 6–Day 10 | Beginner**

This notebook demonstrates zero-shot/few-shot prompts, role/context/instruction/constraints, structured JSON, summarization, classification, generation, coding/debugging, reusable prompt templates, and productivity workflows.

**Run all cells in Demo Mode:** no API key required. Demo outputs are deterministic placeholders/heuristics, not LLM-generated results. Optional live model section is included near the end.


## 1. Setup

In [ ]:
import json, re, textwrap, os
from typing import Any

USE_LIVE_MODEL = False  # Set True only after configuring an API key and installing openai.
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
print("Toolkit ready. Demo mode:", not USE_LIVE_MODEL)


## 2. Prompt anatomy: role, context, instruction, constraints, output format

In [ ]:
def build_prompt(role, context, instruction, constraints=None, output_format=None):
    sections = [f"ROLE:\n{role}", f"CONTEXT:\n{context}", f"INSTRUCTION:\n{instruction}"]
    if constraints:
        if isinstance(constraints, (list, tuple)):
            constraints = "\n".join(f"- {item}" for item in constraints)
        sections.append(f"CONSTRAINTS:\n{constraints}")
    if output_format:
        sections.append(f"OUTPUT FORMAT:\n{output_format}")
    return "\n\n".join(sections)

sample_prompt = build_prompt(
    role="You are a concise study assistant.",
    context="The reader is a beginner learning prompt engineering.",
    instruction="Explain zero-shot prompting.",
    constraints=["Use simple language", "Give one example", "Avoid unsupported claims"],
    output_format="Definition + example"
)
print(sample_prompt)


## 3. Zero-shot vs few-shot prompting

In [ ]:
def zero_shot_prompt(task, context="", constraints=None, output_format=None):
    return build_prompt("You are a helpful assistant.", context or "No extra context.", task, constraints, output_format)

def few_shot_prompt(task, examples, context="", constraints=None, output_format=None):
    examples_text = "\n\n".join(
        f"Example {i+1}\nInput: {ex['input']}\nOutput: {ex['output']}"
        for i, ex in enumerate(examples)
    )
    context = (context + "\n\n" if context else "") + "EXAMPLES:\n" + examples_text
    return build_prompt("Follow the examples carefully.", context, task, constraints, output_format)

zero = zero_shot_prompt("Classify sentiment as positive, negative, or neutral.", "Review: The delivery was quick.")
few = few_shot_prompt("Classify sentiment.", [
    {"input":"I love this product.","output":"positive"},
    {"input":"It arrived broken.","output":"negative"}
], context="Use only the labels positive, negative, neutral.", output_format="One label")
print("ZERO-SHOT:\n", zero, "\n\nFEW-SHOT:\n", few)


## 4. Structured outputs (JSON)

In [ ]:
def extract_json_object(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        raise ValueError("No JSON object found")
    return json.loads(match.group(0))

json_response = '{"category":"billing","priority":"high","summary":"Customer requests invoice correction"}'
parsed = extract_json_object(json_response)
assert set(parsed) == {"category", "priority", "summary"}
print(json.dumps(parsed, indent=2))


## 5. Summarization workflow

In [ ]:
def summarize_demo(text, max_sentences=3):
    sentences = [s.strip() for s in text.replace("\n", " ").split(".") if s.strip()]
    return ". ".join(sentences[:max_sentences]) + ("." if sentences else "")

source = "Prompt design makes instructions clearer. Examples can guide output. Constraints help control format and scope. Evaluation checks whether prompts work."
print(summarize_demo(source, 3))
print("\nPrompt template:\n", build_prompt("Accurate summarizer", source, "Summarize in 3 bullets.", ["Do not add facts"], "3 bullets"))


## 6. Classification workflow

In [ ]:
def classify_demo(text):
    t = text.lower()
    positive = any(w in t for w in ["great", "good", "love", "excellent", "happy", "helpful"])
    negative = any(w in t for w in ["bad", "poor", "hate", "broken", "disappointed", "slow"])
    label = "positive" if positive and not negative else "negative" if negative and not positive else "neutral"
    return {"label": label, "reason": "Keyword-based demo only; not a language model."}

reviews = ["Excellent quality and fast delivery.", "The item was broken and disappointing.", "It is an ordinary product."]
for review in reviews:
    print(review, "=>", classify_demo(review))


## 7. Content generation workflow

In [ ]:
def generate_demo(topic, audience, tone="friendly", format_name="short paragraph"):
    return f"[Demo output] A {tone} {format_name} about {topic} for {audience}. Live LLM generation is optional."

generation_prompt = build_prompt(
    "You are a content writer.",
    "Audience: beginner students",
    "Create an explanation of prompt engineering.",
    ["Keep it accurate", "Use a friendly tone"],
    "A 100-word paragraph"
)
print(generation_prompt)
print(generate_demo("prompt engineering", "beginner students"))


## 8. AI-assisted coding and debugging

In [ ]:
def coding_debug_prompt(code, error, expected_behavior, language="Python"):
    return build_prompt(
        f"You are an experienced {language} developer.",
        f"Code:\n```{language}\n{code}\n```\nError: {error}\nExpected behavior: {expected_behavior}",
        "Find the likely cause and propose a corrected version.",
        ["Explain the fix", "Avoid unrelated changes", "State assumptions"],
        "Diagnosis, corrected code, explanation, test"
    )

debug_prompt = coding_debug_prompt("def divide(a,b): return a/b", "ZeroDivisionError when b is 0", "Return a clear error or safe result.")
print(debug_prompt)


## 9. AI Productivity Toolkit

In [ ]:
def email_draft_prompt(purpose, recipient, context, tone="professional"):
    return build_prompt("You are an email-writing assistant.", f"Recipient: {recipient}\nContext: {context}",
        f"Write an email for: {purpose}", ["Do not invent details", "Keep concise", f"Tone: {tone}"], "Subject + body")

def resume_bullet_prompt(role, experience, achievement):
    return build_prompt("You are a resume-writing assistant.", f"Target role: {role}\nExperience: {experience}\nAchievement: {achievement}",
        "Create 2-4 ATS-friendly resume bullets.", ["Do not fabricate metrics or credentials"], "Bullets")

def research_prompt(question, sources):
    return build_prompt("You are a research assistant.", f"Question: {question}\nSources: {sources}",
        "Prepare a brief with findings.", ["Use only supplied sources", "Separate evidence from uncertainty"], "Overview, findings, limitations")

print(email_draft_prompt("Request a meeting", "Hiring manager", "Ask about interview availability"))
print("\n", resume_bullet_prompt("AI intern", "Student project", "Built a chatbot prototype"))
print("\n", research_prompt("What are prompt evaluation methods?", "Source A: prompt testing overview"))


## 10. Prompt evaluation: reusable test cases

In [ ]:
def evaluate_prompt_tests():
    tests = [
        ("prompt has role", "ROLE:" in sample_prompt),
        ("prompt has context", "CONTEXT:" in sample_prompt),
        ("prompt has instruction", "INSTRUCTION:" in sample_prompt),
        ("constraints included", "CONSTRAINTS:" in sample_prompt),
        ("output format included", "OUTPUT FORMAT:" in sample_prompt),
        ("JSON parse works", extract_json_object(json_response)["priority"] == "high"),
        ("classifier label allowed", classify_demo("great work")["label"] in {"positive","negative","neutral"}),
        ("summary returns text", isinstance(summarize_demo(source), str)),
    ]
    return [{"test": name, "passed": bool(result)} for name, result in tests]

results = evaluate_prompt_tests()
print(json.dumps(results, indent=2))
assert all(item["passed"] for item in results), "One or more prompt tests failed."
print(f"Passed {sum(r['passed'] for r in results)}/{len(results)} checks.")


## 11. Optional live model (OpenAI-compatible example)

In [ ]:
# Optional: Uncomment/install and configure a key securely in Colab Secrets.
# !pip -q install openai
#
# from google.colab import userdata
# api_key = userdata.get("OPENAI_API_KEY")  # Store key in Colab Secrets, never in notebook.
# from openai import OpenAI
# client = OpenAI(api_key=api_key)
# response = client.chat.completions.create(
#     model=MODEL_NAME,
#     messages=[{"role":"user","content":sample_prompt}],
#     temperature=0.2
# )
# print(response.choices[0].message.content)
#
# Record live outputs and human scores separately. Model access and pricing depend on your account.


## 12. Reflection and next steps

- Zero-shot prompting is useful when the task is clear and examples are unnecessary.
- Few-shot prompting provides demonstrations that can help guide output patterns.
- Structured output instructions make downstream parsing easier, but outputs should still be validated.
- Role/context/instruction/constraints make prompts easier to inspect and reuse.
- Automated checks verify basic formatting and workflow behavior; human evaluation is still needed for factuality, usefulness, and nuanced quality.
- Next improvement: run the same evaluation cases against several prompt variants and a live model, then compare rubric scores.
